# Did IBM's flagship quantum chemistry benchmark compute the state it claims?

**Press the play button on the cell below. It takes about two seconds, downloads nothing, and installs nothing.**

Sample-based quantum diagonalization (SQD) is behind one of the flagship "quantum computers are useful for
chemistry now" results: IBM's iron-sulfur cluster simulations in *Science Advances* (2025). A published
critique (arXiv:2501.07231) argues classical selected-CI wins at matched cost. That argument has been running
for a year, entirely about energies.

Nobody measured which electronic *state* those energies belong to.

The target is a **singlet**, which has total spin squared &#10216;S&#178;&#10217; = 0. This notebook reads two
archives produced by IBM's own shipped pipeline (`qiskit-addon-sqd`), run with their spin-completion
mitigation off and then on, and reports what actually came back.

The two archives are embedded in this notebook as base64. Their SHA-256 hashes are printed when you run it, and
they match the files in the public archive at
[doi:10.5281/zenodo.21359923](https://doi.org/10.5281/zenodo.21359923), so you can verify these are the same
bytes and not something cooked up for a demo.


In [ ]:
# Self-contained. numpy is the only dependency; Colab already has it.
import base64, hashlib, io, platform
import numpy as np

BLOBS = {
    "off": {"b64": "UEsDBC0AAAAAAAAAIQBslIOU//////////8IABQAaGlzdC5ucHkBABAA2AQAAAAAAADYBAAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJ3xPJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDEyLCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKgASVTQQAAAAAAACMFm51bXB5Ll9jb3JlLm11bHRpYXJyYXmUjAxfcmVjb25zdHJ1Y3SUk5SMBW51bXB5lIwHbmRhcnJheZSTlEsAhZRDAWKUh5RSlChLAUsMhZRoA4wFZHR5cGWUk5SMAk84lImIh5RSlChLA4wBfJROTk5K/////0r/////Sz90lGKJXZQofZQojARpdGVylEsBjAViYXRjaJRLAIwFZGltX2GUS32MBWRpbV9ilEt3jAVlX3RvdJRoAIwGc2NhbGFylJOUaAyMAmY4lImIh5RSlChLA4wBPJROTk5K/////0r/////SwB0lGJDCKXm/06KIF3AlIaUUpSMB2Vycl9tSGGUaBpoHUMIwAn05rZKWECUhpRSlIwCczKUR0ATiSz4LRgSdX2UKGgUSwFoFUsBaBZLemgXS3toGGgaaB1DCEnakgD+IF3AlIaUUpRoI2gaaB1DCCBR7EDJhlZAlIaUUpRoJ0dAE50DTQokJnV9lChoFEsBaBVLAmgWS3hoF0t3aBhoGmgdQwhRMfTg7CBdwJSGlFKUaCNoGmgdQwjgWbDErMlWQJSGlFKUaCdHQBOi+eNWyFh1fZQoaBRLAmgVSwBoFkutaBdLrGgYaBpoHUMIZCQ76MwiXcCUhpRSlGgjaBpoHUMIUK8VryDtTkCUhpRSlGgnR0ATfy+4VsRhdX2UKGgUSwJoFUsBaBZLpmgXS6doGGgaaB1DCNZrPg2rIl3AlIaUUpRoI2gaaB1DCLCEd4Wf9U9AlIaUUpRoJ0dAE4QyusIAqHV9lChoFEsCaBVLAmgWS6toF0ulaBhoGmgdQwjBUTvcqSJdwJSGlFKUaCNoGmgdQwjASLNt7v5PQJSGlFKUaCdHQBN8G6/5xdJ1fZQoaBRLA2gVSwBoFkvCaBdLxmgYaBpoHUMIsZqdzXgjXcCUhpRSlGgjaBpoHUMIwHXZnTCuSUCUhpRSlGgnR0ATZ/8K+LN0dX2UKGgUSwNoFUsBaBZLy2gXS8NoGGgaaB1DCEQMKtlQI13AlIaUUpRoI2gaaB1DCFApomRW5kpAlIaUUpRoJ0dAE2q0YW0T03V9lChoFEsDaBVLAmgWS8loF0vJaBhoGmgdQwhNd4tNYyNdwJSGlFKUaCNoGmgdQwgA840rKVZKQJSGlFKUaCdHQBNdaKm9t1p1fZQoaBRLBGgVSwBoFkvYaBdL4WgYaBpoHUMI9a+sy9gjXcCUhpRSlGgjaBpoHUMIgFIDyD/ARkCUhpRSlGgnR0ATVleKRDmsdX2UKGgUSwRoFUsBaBZL1WgXS9xoGGgaaB1DCD+Y9t3NI13AlIaUUpRoI2gaaB1DCGCQnOagFUdAlIaUUpRoJ0dAE1sGWO9P7nV9lChoFEsEaBVLAmgWS9loF0viaBhoGmgdQwiC1UPl1yNdwJSGlFKUaCNoGmgdQwjw9C3bR8dGQJSGlFKUaCdHQBNZh7cD5CJ1ZXSUYi5QSwMELQAAAAAAAAAhADEJYzr//////////woAFABiZXN0X2UubnB5AQAQAIgAAAAAAAAAiAAAAAAAAACTTlVNUFkBAHYAeydkZXNjcic6ICc8ZjgnLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoKSwgfSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgCvWvrMvYI13AUEsDBC0AAAAAAAAAIQCxp9QH//////////8LABQAYmVzdF9zMi5ucHkBABAAiAAAAAAAAACIAAAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxmOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKrDlEildWE0BQSwECLQMtAAAAAAAAACEAbJSDlNgEAADYBAAACAAAAAAAAAAAAAAAgAEAAAAAaGlzdC5ucHlQSwECLQMtAAAAAAAAACEAMQljOogAAACIAAAACgAAAAAAAAAAAAAAgAESBQAAYmVzdF9lLm5weVBLAQItAy0AAAAAAAAAIQCxp9QHiAAAAIgAAAALAAAAAAAAAAAAAACAAdYFAABiZXN0X3MyLm5weVBLBQYAAAAAAwADAKcAAACbBgAAAAA=", "sha256": "3b4225b562db9e3381afbe9812ba7c375e74eb8ed4cfc6f07d6afdae044730e4"},   # symmetrize_spin = False
    "on":  {"b64": "UEsDBC0AAAAAAAAAIQCMea68//////////8IABQAaGlzdC5ucHkBABAA6gQAAAAAAADqBAAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJ3xPJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDEyLCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKgASVXwQAAAAAAACMFm51bXB5Ll9jb3JlLm11bHRpYXJyYXmUjAxfcmVjb25zdHJ1Y3SUk5SMBW51bXB5lIwHbmRhcnJheZSTlEsAhZRDAWKUh5RSlChLAUsMhZRoA4wFZHR5cGWUk5SMAk84lImIh5RSlChLA4wBfJROTk5K/////0r/////Sz90lGKJXZQofZQojARpdGVylEsBjAViYXRjaJRLAIwFZGltX2GUS/SMBWRpbV9ilEv0jAVlX3RvdJRoAIwGc2NhbGFylJOUaAyMAmY4lImIh5RSlChLA4wBPJROTk5K/////0r/////SwB0lGJDCNPm/06KIF3AlIaUUpSMB2Vycl9tSGGUaBpoHUMIEFbz5rZKWECUhpRSlIwCczKUR0ATiSz4L3yMdX2UKGgUSwFoFUsBaBZL9WgXS/VoGGgaaB1DCNHZkgD+IF3AlIaUUpRoI2gaaB1DCOAl7kDJhlZAlIaUUpRoJ0dAE50DTQoTInV9lChoFEsBaBVLAmgWS+9oF0vvaBhoGmgdQwhKMfTg7CBdwJSGlFKUaCNoGmgdQwg4dbDErMlWQJSGlFKUaCdHQBOi+eNWwbp1fZQoaBRLAmgVSwBoFk1ZAWgXTVkBaBhoGmgdQwhkJDvozCJdwJSGlFKUaCNoGmgdQwhQrxWvIO1OQJSGlFKUaCdHQBN/L7hWhxd1fZQoaBRLAmgVSwFoFk1NAWgXTU0BaBhoGmgdQwi/az4NqyJdwJSGlFKUaCNoGmgdQwhgOHiFn/VPQJSGlFKUaCdHQBOEMrrBBSV1fZQoaBRLAmgVSwJoFk1QAWgXTVABaBhoGmgdQwjgUTvcqSJdwJSGlFKUaCNoGmgdQwiQVrJt7v5PQJSGlFKUaCdHQBN8G6/5rZp1fZQoaBRLA2gVSwBoFk2IAWgXTYgBaBhoGmgdQwj3mp3NeCNdwJSGlFKUaCNoGmgdQwjgUtedMK5JQJSGlFKUaCdHQBNn/wr4XaV1fZQoaBRLA2gVSwFoFk2OAWgXTY4BaBhoGmgdQwjcDCrZUCNdwJSGlFKUaCNoGmgdQwjQhZ1kVuZKQJSGlFKUaCdHQBNqtGFtD+51fZQoaBRLA2gVSwJoFk2SAWgXTZIBaBhoGmgdQwhKd4tNYyNdwJSGlFKUaCNoGmgdQwhwCo4rKVZKQJSGlFKUaCdHQBNdaKm9t6Z1fZQoaBRLBGgVSwBoFk25AWgXTbkBaBhoGmgdQwgfsKzL2CNdwJSGlFKUaCNoGmgdQwhgCgLIP8BGQJSGlFKUaCdHQBNWV4pErMV1fZQoaBRLBGgVSwFoFk2xAWgXTbEBaBhoGmgdQwgomPbdzSNdwJSGlFKUaCNoGmgdQwgQRJ3moBVHQJSGlFKUaCdHQBNbBljvwA11fZQoaBRLBGgVSwJoFk27AWgXTbsBaBhoGmgdQwiR1UPl1yNdwJSGlFKUaCNoGmgdQwjAfy3bR8dGQJSGlFKUaCdHQBNZh7cEGdZ1ZXSUYi5QSwMELQAAAAAAAAAhAL/c5bn//////////woAFABiZXN0X2UubnB5AQAQAIgAAAAAAAAAiAAAAAAAAACTTlVNUFkBAHYAeydkZXNjcic6ICc8ZjgnLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoKSwgfSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgCh+wrMvYI13AUEsDBC0AAAAAAAAAIQC6RYB1//////////8LABQAYmVzdF9zMi5ucHkBABAAiAAAAAAAAACIAAAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxmOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKxaxEildWE0BQSwECLQMtAAAAAAAAACEAjHmuvOoEAADqBAAACAAAAAAAAAAAAAAAgAEAAAAAaGlzdC5ucHlQSwECLQMtAAAAAAAAACEAv9zluYgAAACIAAAACgAAAAAAAAAAAAAAgAEkBQAAYmVzdF9lLm5weVBLAQItAy0AAAAAAAAAIQC6RYB1iAAAAIgAAAALAAAAAAAAAAAAAACAAegFAABiZXN0X3MyLm5weVBLBQYAAAAAAwADAKcAAACtBgAAAAA=",  "sha256": "7cd9206b850e64b7cdac0b02e3113fbf67946ed7a08dcc01d02db88a7dbb8383"},    # symmetrize_spin = True
}

arch = {}
print("# integrity of the embedded archives")
for tag, rec in BLOBS.items():
    raw = base64.b64decode(rec["b64"])
    got = hashlib.sha256(raw).hexdigest()
    ok = "OK" if got == rec["sha256"] else "MISMATCH"
    print(f"  completion {tag:3s}  {len(raw):>6,} bytes  sha256 {got[:16]}...  {ok}")
    arch[tag] = np.load(io.BytesIO(raw), allow_pickle=True)

print(f"\n# environment\n  python {platform.python_version()}   numpy {np.__version__}")

def maxdim(hist):
    """Largest subspace actually diagonalized, over the recovery iterations."""
    return max(int(r["dim_a"]) * int(r["dim_b"]) for r in hist)

e_off, e_on = float(arch["off"]["best_e"]), float(arch["on"]["best_e"])
s2_off, s2_on = float(arch["off"]["best_s2"]), float(arch["on"]["best_s2"])
d_off, d_on = maxdim(list(arch["off"]["hist"])), maxdim(list(arch["on"]["hist"]))

print("\n# as-shipped pipeline on the [2Fe-2S] samples (seed 17)")
print("# the singlet target these papers name has <S^2> = 0")
print(f"{'':28s}{'best energy (Ha)':>20s}{'<S^2>':>12s}{'determinants':>16s}")
print(f"{'BEFORE completion (off)':28s}{e_off:>20.8f}{s2_off:>12.5f}{d_off:>16,}")
print(f"{'AFTER  completion (on)':28s}{e_on:>20.8f}{s2_on:>12.5f}{d_on:>16,}")
print(f"{'delta':28s}{(e_on - e_off) * 1e9:>17.3f} nHa{s2_on - s2_off:>12.1e}{d_on / d_off:>15.2f}x")

checks = [
    ("completion moved the energy by less than 1 nHa", abs(e_on - e_off) < 1e-9),
    ("completion left <S^2> essentially unchanged", abs(s2_on - s2_off) < 1e-6),
    ("completion multiplied the determinant count ~4x", 3.9 < d_on / d_off < 4.1),
    ("the returned state is a high-spin mixture, not a singlet", s2_on > 3.0),
]
print("\nVERDICT")
for label, ok in checks:
    print(f"  {label:.<62s} {'PASS' if ok else 'FAIL'}")

print(f"""
Spin completion quadrupled the subspace ({d_off:,} -> {d_on:,} determinants) and
changed the ground-state energy by {abs(e_on - e_off) * 1e9:.3f} nHa. The returned state sits at
<S^2> = {s2_on:.3f}. The singlet these benchmarks target is at 0.

It makes a singlet representable. It never produces one.""")


## What you just saw

Spin completion is the mitigation the flagship work relies on to handle spin. Run as shipped, it multiplies the
number of determinants by four and moves the ground-state energy by less than a nanohartree, while the spin of
the returned state does not move at all. The state comes back at &#10216;S&#178;&#10217; near 4.83, not 0.

The reason is structural rather than a tuning failure: over the completed ground manifold the S&#178; block comes
out proportional to the identity, so every state in that manifold carries the same contamination. Completion
opens the door to a singlet without ever preferring one.

The spin ladder in these systems spans roughly 15 to 30 mHa, which is the same size as the accuracy
improvements the two sides of this dispute are arguing about.

## Which number is this? (the paper reports three)

All three are [2Fe-2S], and they measure different things. Reading them as one number in disagreement with
itself is the mistake this notebook exists to prevent.

- **&#10216;S&#178;&#10217; = 4.83, above.** What IBM's shipped pipeline returns here, after four recovery
  iterations, on samples drawn from a converged benchmark state.
- **&#10216;S&#178;&#10217; &#8776; 4.66.** The contaminated attractor that classical selected-CI growth
  approaches on this system under the audited HCI protocols. The pipeline inherits the contamination of the
  state it samples, which is why these two sit close together.
- **&#10216;S&#178;&#10217; = 1.3711.** A different measurement: the numerically stable lowest root at the
  largest *published* [2Fe-2S] dimension (5.625&#215;10&#8311; determinants), in a subspace reconstructed from
  the released support. Its spin variance, Var(S&#178;) = 6.9, excludes any decomposition confined to
  S &#8804; 2, so the smaller mean is not a near-singlet either.

The lower mean being just as contaminated is itself one of the paper's findings: a scalar
&#10216;S&#178;&#10217; does not identify a state. Two audited arms both sit at &#10216;S&#178;&#10217;
&#8776; 2.00, and one is a clean triplet while the other is a 2:1 singlet-quintet mixture. None of the three
numbers above is the singlet these papers name.

## Going further

This is the smallest of three reproducers. The other two, plus every number in the paper, are in the public
archive:

- **Paper:** [doi:10.26434/chemrxiv.15006382/v1](https://doi.org/10.26434/chemrxiv.15006382/v1)
- **Data, code, audit trail:** [doi:10.5281/zenodo.21359923](https://doi.org/10.5281/zenodo.21359923)
- `REPRO_MAP.md` in the archive maps every claim in the paper to the file and command that regenerates it.
- `_paperaudit.py` re-derives all 370 quantitative claims from the raw archives and fails if any one drifts.

## How to show this is wrong

Any one of these would overturn the central claim, and the instruments for all three ship in the archive:

1. Exhibit any state in a spin-completed ground manifold of these benchmarks with &#10216;S&#178;&#10217; &lt; 1.
2. Produce a quantum-sampled subspace at matched determinant count whose spin-identified energy beats HCI or CIPSI.
3. Get any run of the shipped pipeline on IBM's archived samples to land &#10216;S&#178;&#10217; &lt; 1 and error under
   50 mHa at the same time, with no spin penalty.

If someone does any of it, it gets published, including if that means I was wrong.

*Tyler Vitale, Pure State Labs*
